In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("StealthPhisher2025.csv")
df.iloc[:,-1:] = df.iloc[:,-1:].replace(to_replace ="Phishing", value =0)
df.iloc[:,-1:] = df.iloc[:,-1:].replace(to_replace ="Benign", value =1)
LABEL = df.iloc[:,-1:].columns[0]
df[LABEL] = df[LABEL].apply(lambda x: int(x))
colsAll = df.select_dtypes(include=['float64','int64']).columns
colsAll.append(pd.Index([LABEL]))
df = pd.DataFrame(df[colsAll]).copy()
print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
df.head(5)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
import pandas as pd
import numpy as np

def calculate_mrmr_ranking(dfMRMR):
    X = dfMRMR.drop(columns=[LABEL])

    # Step 1: Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    # Step 2: Calculate pairwise mutual information
    def mutual_information(x, y):
        return mutual_info_regression(x.values.reshape(-1, 1), y.values.ravel())[0]
    
    n_features = X_scaled.shape[1]
    mutual_info_matrix = np.zeros((n_features, n_features))
    
    for i in range(n_features):
        for j in range(n_features):
            if i == j:
                mutual_info_matrix[i, j] = 0  # Ignore self-relevance
            else:
                mutual_info_matrix[i, j] = mutual_information(X_scaled.iloc[:, i], X_scaled.iloc[:, j])

    # Step 3: Calculate mRMR score for each feature
    scores = {}
    for i, feature in enumerate(X_scaled.columns):
        relevance = np.mean(mutual_info_matrix[i, :])  # Mean relevance to other features
        redundancy = np.mean(mutual_info_matrix[:, i])  # Mean redundancy of this feature
        scores[feature] = relevance - redundancy  # mRMR score

    # Step 4: Rank features based on scores
    ranked_features = pd.Series(scores).sort_values(ascending=False)
    return ranked_features

def calculate_pca_ranking(dfPCA):
    X = dfPCA.drop(columns=[LABEL])  
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)    
    pca = PCA(n_components=X.shape[1])
    pca.fit(X_scaled)
    pca_importance = pd.Series(pca.explained_variance_ratio_, index=X.columns)
    return pca_importance
def plot_feature_ranking(ranking, threshold=0.2, title="Feature Ranking by PCA"):
    print(len(ranking))     
    filtered_ranking = ranking[ranking > threshold]
    print(f"Number of features above threshold: {len(filtered_ranking)}")
    filtered_ranking = filtered_ranking.sort_index()
    plt.figure(figsize=(16, 8))
    plt.plot(filtered_ranking.index, filtered_ranking.values, marker='o', linestyle='-', color='b',linewidth=2.5)
    plt.xlabel("Features",fontsize=15, fontweight='bold')
    plt.ylabel("PCA Explained Variance Ratio",fontsize=15, fontweight='bold')
    plt.xticks(fontsize=14, rotation=75, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(False)
    plt.tight_layout()
    name = 'charts\\'+title + '.png'
    plt.savefig(name, format='png', dpi=300, bbox_inches='tight')
    plt.show()
def plot_pie_chart(ranking, threshold=0.2, title="PCA Feature Importance Pie Chart"):
    ranking = ranking.sort_index()
    filtered_ranking = ranking[ranking > threshold]
    print(f"Number of features above threshold: {len(filtered_ranking)}")
    plt.figure(figsize=(10, 10))
    plt.pie(
        filtered_ranking,
        labels=filtered_ranking.index,
        autopct="%1.1f%%",
        startangle=135,
        pctdistance=0.85,
        labeldistance=1.02,
        colors=plt.cm.tab20.colors,
        textprops={'fontweight': 'bold'}
    )
    #plt.title(title, fontsize=16, fontweight='bold')
    plt.grid(False)
    plt.tight_layout()
    name = 'charts\\FS_Final.png'
    plt.savefig(name, format='png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
class_0_df = df[df[LABEL] == 0]
class_1_df = df[df[LABEL] == 1]
class_0_pca = calculate_pca_ranking(class_0_df)
print('Done Benign')
class_1_pca = calculate_pca_ranking(class_1_df)
print('Done Malware')

In [ ]:
plot_feature_ranking(class_0_pca,threshold=0.002, title="PCA Explained Variance - Legitimate URLs")

In [ ]:
plot_feature_ranking(class_1_pca,threshold=0.002, title="PCA Explained Variance - Malicious URLs")

In [ ]:
# Combine PCA values for both classes
combined_pca = pd.concat([class_0_pca, class_1_pca], axis=1, keys=["Class 0", "Class 1"])
combined_pca["Average"] = combined_pca.mean(axis=1)
# Plot combined PCA values, sorted by feature names
plot_feature_ranking(combined_pca["Average"], threshold=0.01,title="Average PCA Explained Variance for Legitimate and Malicious URLS")

In [ ]:
# Plot pie chart for features with average PCA > threshold
plot_pie_chart(combined_pca["Average"], threshold=0.01, title="Combined PCA Importance (Threshold > 0.15)")

In [ ]:
rslt_df = combined_pca[(combined_pca['Average'] >=0.01)]
print(len(rslt_df.index))
rslt_df.index

In [ ]:
rslt_df = combined_pca[(combined_pca['Average'] >=0.02)]
print(len(rslt_df.index))
rslt_df.index

In [ ]:
combined_pca.to_csv('combined_pca.csv')